# Classification experiments

### Описание эксперимента

На данном этапе необходимо построить и сравнить несколько моделей классификации для диагностики состояния компонентов гидравлической системы.

Мы рассматриваем четыре целевые переменные:

* `cooler` — состояние охладителя;
* `valve` — состояние клапана;
* `pump` — состояние насоса;
* `accumulator` — состояние аккумулятора.

Для каждого target сравниваются три алгоритма:

* Logistic Regression;
* Random Forest;
* HistGradientBoosting.

Основная задача эксперимента — не только получить высокую метрику, но и проверить, насколько результат зависит от способа разбиения данных на обучающую и тестовую выборки.

Это особенно важно для данного датасета, поскольку наблюдения представляют собой последовательные циклы работы одной системы. Соседние циклы могут быть очень похожи друг на друга, поэтому случайное разбиение способно давать слишком оптимистичную оценку качества.

Поэтому далее рассматриваются три различных сценария валидации.


In [ ]:
from pathlib import Path

import numpy as np  # noqa: F401
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Загрузка подготовленных данных

In [ ]:
DATA_DIR = Path("../data/processed")

FEATURES_PATH = DATA_DIR / "X.csv"
PROFILE_PATH = DATA_DIR / "y.csv"

features = pd.read_csv(FEATURES_PATH)
profile = pd.read_csv(PROFILE_PATH)

print("features:", features.shape)
print("profile: ", profile.shape)

display(features.head())
display(profile.head())

In [ ]:
TARGETS = [
    "cooler",
    "valve",
    "pump",
    "accumulator",
]

X = features.drop(columns=["cycle_id"], errors="ignore").copy()
y_all = profile[TARGETS].copy()

print(f"Samples:  {len(X)}")
print(f"Features: {X.shape[1]}")

## 2. Состав эксперимента

In [ ]:
models = {
    "logistic_regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=42,
                ),
            ),
        ]
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.1,
        max_leaf_nodes=31,
        random_state=42,
    ),
}

MODEL_LABELS = {
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
    "hist_gradient_boosting": "HistGradientBoosting",
}

TARGET_LABELS = {
    "cooler": "Cooler",
    "valve": "Valve",
    "pump": "Pump",
    "accumulator": "Accumulator",
}

## 3. Split №1 — random stratified

Первый вариант - стандартное случайное стратифицированное разбиение. Этот вариант нужен прежде всего как **baseline для сравнения**.

Однако для нашего датасета он не является наиболее надёжной оценкой. Циклы идут последовательно, и соседние наблюдения могут быть очень похожи. Поэтому после случайного разбиения похожие циклы могут одновременно оказаться в train и test.

Следовательно, высокая метрика на таком split ещё не означает хорошую способность модели обобщаться на новые экспериментальные условия.


In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

random_results = []

for target in TARGETS:
    y = profile[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        random_results.append(
            {
                "split": "random_stratified",
                "target": target,
                "model": model_name,
                "accuracy": accuracy_score(y_test, y_pred),
                "f1_macro": f1_score(
                    y_test,
                    y_pred,
                    average="macro",
                ),
                "f1_weighted": f1_score(
                    y_test,
                    y_pred,
                    average="weighted",
                ),
            }
        )

random_results_df = pd.DataFrame(random_results)

display(random_results_df.sort_values(["target", "f1_macro"], ascending=[True, False]).round(4))

Как и стоило ожидать - результаты очень высокие для всех таргетов, где-то аж единица

## 4. Split №2 — experimental blocks

Попробуем испраавить ситуацию, при которой последовательные циклы попадают в train и test. 

Учтем структуру самих данных и объединим последовательные циклы в блоки.

Блок — это непрерывная последовательность циклов, внутри которой не меняется
ни один из четырёх режимов:

`cooler × valve × pump × accumulator`

Таким образом, один экспериментальный блок целиком попадает только в один fold.

Например, если несколько последовательных циклов были записаны при одинаковых состояниях всех четырёх компонентов, они не должны искусственно разделяться между train и test.

Такой подход лучше отражает структуру исходного эксперимента и уменьшает влияние зависимости между соседними наблюдениями.

In [ ]:
regime = profile[["cooler", "valve", "pump", "accumulator"]]

experimental_groups = regime.ne(regime.shift()).any(axis=1).cumsum()

block_sizes = experimental_groups.value_counts().sort_index()

print("Number of blocks:", experimental_groups.nunique())
print("\nBlock sizes:")
print(block_sizes.value_counts().sort_index())

print("\nFirst blocks:")
display(
    pd.DataFrame(
        {
            "block": experimental_groups,
            "cooler": profile["cooler"],
            "valve": profile["valve"],
            "pump": profile["pump"],
            "accumulator": profile["accumulator"],
        }
    )
    .groupby("block")
    .agg(
        start_cycle=("block", lambda x: x.index.min()),
        end_cycle=("block", lambda x: x.index.max()),
        size=("block", "size"),
        cooler=("cooler", "first"),
        valve=("valve", "first"),
        pump=("pump", "first"),
        accumulator=("accumulator", "first"),
    )
    .head(15)
)

#### Cross-validation по экспериментальным блокам

Для разделения используем StratifiedGroupKFold.

Здесь одновременно учитываются два требования:

- Stratification — стараемся сохранить распределение классов target между fold'ами.
- Grouping — один экспериментальный блок не может попасть одновременно в train и test.

Используем 4 fold'а

Для каждого fold модель обучается на экспериментальных блоках train-части и проверяется на полностью отложенных блоках.

После этого все объекты получают out-of-fold предсказания, поскольку каждый объект предсказывается моделью, которая не обучалась на этом объекте.

In [ ]:
experimental_cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42,
)

experimental_fold_results = []
experimental_oof_results = []

for target in TARGETS:
    y = profile[target]

    for model_name, model in models.items():
        oof_predictions = pd.Series(
            index=X.index,
            dtype=y.dtype,
        )

        for fold, (train_idx, test_idx) in enumerate(
            experimental_cv.split(
                X,
                y,
                groups=experimental_groups,
            ),
            start=1,
        ):
            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            oof_predictions.iloc[test_idx] = y_pred

            experimental_fold_results.append(
                {
                    "split": "experimental_blocks",
                    "target": target,
                    "model": model_name,
                    "fold": fold,
                    "accuracy": accuracy_score(y_test, y_pred),
                    "f1_macro": f1_score(
                        y_test,
                        y_pred,
                        average="macro",
                    ),
                    "f1_weighted": f1_score(
                        y_test,
                        y_pred,
                        average="weighted",
                    ),
                }
            )

        assert oof_predictions.notna().all()

        experimental_oof_results.append(
            {
                "split": "experimental_blocks",
                "target": target,
                "model": model_name,
                "f1_macro_overall": f1_score(
                    y,
                    oof_predictions,
                    average="macro",
                ),
                "f1_weighted_overall": f1_score(
                    y,
                    oof_predictions,
                    average="weighted",
                ),
            }
        )

experimental_fold_results_df = pd.DataFrame(experimental_fold_results)

experimental_oof_results_df = pd.DataFrame(experimental_oof_results)

print("Mean fold metrics:")
display(
    experimental_fold_results_df.groupby(["target", "model"])[
        ["accuracy", "f1_macro", "f1_weighted"]
    ]
    .agg(["mean", "std"])
    .round(4)
)

print("\nOverall OOF metrics:")
display(
    experimental_oof_results_df.sort_values(
        ["target", "f1_macro_overall"],
        ascending=[True, False],
    ).round(4)
)

## 5. Split №3 — unseen cooler × accumulator combinations

Здесь группа определяется парой:

`cooler × accumulator`

Это намеренно более строгий сценарий: некоторые комбинации режимов полностью
исключаются из обучения конкретного fold. Такой split проверяет способность
модели обобщаться на невидимые комбинации состояний.

Этот эксперимент отвечает на иной вопрос: насколько хорошо модель может работать на комбинациях состояний cooler и accumulator, которых она не видела при обучении?

Это уже не просто проверка интерполяции между похожими экспериментами, а более сложный тест способности модели обобщаться на новые режимы работы системы.


In [ ]:
regime_groups = profile["cooler"].astype(str) + "_" + profile["accumulator"].astype(str)

print("Number of cooler × accumulator groups:")
print(regime_groups.nunique())

display(profile.assign(group=regime_groups).groupby("group").size().rename("samples").to_frame())

In [ ]:
regime_cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42,
)

regime_results = []

for target in TARGETS:
    y = profile[target]

    for model_name, model in models.items():
        for fold, (train_idx, test_idx) in enumerate(
            regime_cv.split(
                X,
                y,
                groups=regime_groups,
            ),
            start=1,
        ):
            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            regime_results.append(
                {
                    "split": "cooler_x_accumulator",
                    "target": target,
                    "model": model_name,
                    "fold": fold,
                    "accuracy": accuracy_score(y_test, y_pred),
                    "f1_macro": f1_score(
                        y_test,
                        y_pred,
                        average="macro",
                    ),
                    "f1_weighted": f1_score(
                        y_test,
                        y_pred,
                        average="weighted",
                    ),
                }
            )

regime_results_df = pd.DataFrame(regime_results)

regime_summary_df = regime_results_df.groupby(["target", "model"])[
    ["accuracy", "f1_macro", "f1_weighted"]
].agg(["mean", "std"])

display(regime_summary_df.round(4))

## 6. Сводное сравнение

Теперь сравниваем качество моделей при трёх различных сценариях валидации.

Важно, что сами признаки и алгоритмы во всех экспериментах остаются одинаковыми. Меняется только способ формирования train/test.

Это позволяет отделить два эффекта:

- качество самой модели;
- влияние предположений о независимости наблюдений и структуре будущих данных.

Если random split показывает значительно более высокую F1, чем grouped split, это означает, что случайная валидация, вероятно, даёт слишком оптимистичную оценку для данной структуры данных.

In [ ]:
# Средние F1 macro по fold'ам для grouped split.
experimental_summary = (
    experimental_fold_results_df.groupby(["target", "model"])["f1_macro"]
    .mean()
    .rename("f1_macro")
    .reset_index()
)

regime_summary = (
    regime_results_df.groupby(["target", "model"])["f1_macro"]
    .mean()
    .rename("f1_macro")
    .reset_index()
)

random_summary = random_results_df[["target", "model", "f1_macro"]].copy()

random_summary["split"] = "random_stratified"
experimental_summary["split"] = "experimental_blocks"
regime_summary["split"] = "cooler_x_accumulator"

comparison_df = pd.concat(
    [
        random_summary,
        experimental_summary,
        regime_summary,
    ],
    ignore_index=True,
)

comparison_table = comparison_df.pivot_table(
    index=["target", "model"],
    columns="split",
    values="f1_macro",
).reset_index()

display(comparison_table.round(4))

### Сравнение всех моделей по каждому target

Для каждого target можно отдельно посмотреть, какая модель выигрывает
при каждом из трёх сценариев.


In [ ]:
for target in TARGETS:
    print(f"{TARGET_LABELS[target]}")

    display(
        comparison_df[comparison_df["target"] == target]
        .pivot(
            index="model",
            columns="split",
            values="f1_macro",
        )
        .rename(index=MODEL_LABELS)
        .round(4)
    )

## 7. Финальная оценка experimental-block split

Для выбранного основного split полезно различать два варианта F1:

- **mean fold F1** — среднее F1, полученное на четырёх fold'ах;
- **overall OOF F1** — все out-of-fold предсказания объединяются и F1 считается
  один раз на всех 2205 объектах.

Для финальной единой цифры по cross-validation используем **overall OOF F1 macro**.


> Overall OOF F1

Помимо среднего значения F1 по отдельным fold'ам, есть смысл рассчитать единую F1 на всех out-of-fold предсказаниях.

Если
$
\hat y_i^{OOF}
$
— предсказание для объекта \(i\), полученное моделью, которая не использовала этот объект при обучении, то после объединения предсказаний всех четырёх fold'ов получаем один полный набор $\hat{\mathbf y}^{OOF}.$

Затем вычисляем `F1 macro`, где каждый класс получает одинаковый вес.

Также считаем `F1 weighted`, где вклад каждого класса пропорционален количеству его объектов.


In [ ]:
final_oof = experimental_oof_results_df.pivot(
    index="target",
    columns="model",
    values="f1_macro_overall",
).rename(columns=MODEL_LABELS)

display(final_oof.round(4))

In [ ]:
final_oof_weighted = experimental_oof_results_df.pivot(
    index="target",
    columns="model",
    values="f1_weighted_overall",
).rename(columns=MODEL_LABELS)

display(final_oof_weighted.round(4))

## 8. Итог

- `cycle_id` не используется как признак.
- Все три split'а оцениваются на одних и тех же 170 cycle-level признаках.
- Все четыре target оцениваются одинаковым набором моделей.
- Random split служит базовым ориентиром.
- Experimental-block split - основной рабочий сценарий.
- Cooler × accumulator split - стресс-тест на обобщение на невидимые комбинации режимов.
- Для основного сценария итоговая F1 считается по всем OOF-предсказаниям.
